# **Fine Tune With Adapters**

#### **Import Libraries**

In [1]:
import torch
import torch.nn as nn 
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, BertForSequenceClassification, BertTokenizer
from tqdm import tqdm
from torch.utils.data import Dataset
import math
from torch.utils.data import random_split 
from torch.utils.data import DataLoader

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### **Dataset Configuration**

In [2]:
dataset = load_dataset("imdb")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [3]:
train_dataset = dataset['train']
test_dataset = dataset['test']

In [4]:
train_dataset[0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [5]:
label_list = []
for id , (text, label) in enumerate(train_dataset):
    label_list.append(train_dataset[id]['label'])

num_labels = len(set(label_list))
num_labels    

2

In [6]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
tokenizer

BertTokenizer(name_or_path='bert-base-cased', vocab_size=28996, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [7]:
lmdb_labels = {0: "negatice review", 1: "positive review"}

In [8]:
label_col = train_dataset.select_columns('label')
label_col

Dataset({
    features: ['label'],
    num_rows: 25000
})

In [9]:
len(train_dataset)

25000

In [10]:
imdb_labels = {0: "negative review", 1: "positive review"}

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device.type

'cuda'

### **Define Dataset Class**

In [12]:
class ClassificationDataset(Dataset):
    
    def __init__(self,dataset,tokenizer: AutoTokenizer, max_length:int = 500):
        super().__init__()
        self.max_length = max_length
        self.tokenizer = tokenizer
        self.dataset = dataset
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, index):
        item = self.dataset[index]
        
        encodings = self.tokenizer(
            item['text'],
            return_tensors = 'pt',
            max_length = self.max_length,
            truncation = True,
            padding = 'max_length'
        )
        label_tenosr = torch.tensor(item['label'], dtype=torch.int64)
        return encodings['input_ids'].squeeze(0), encodings['attention_mask'].squeeze(0), label_tenosr
        

In [13]:
dataset_train = ClassificationDataset(train_dataset, tokenizer)
dataset_test = ClassificationDataset(test_dataset, tokenizer)

In [14]:
for i in range(-10,10):
    print(dataset_train[i])

(tensor([  101,  2409,   146,  1163,  1157,   170,  4610,  3774,   119,  1135,
         1218,  1637,  1218,  5376,  1105,  1218,  2641,   119,   146,  3851,
         1917,  1107,  1142,  2523,   119,  4785,  1157,  4613,  1155,  1268,
         1133,  1103, 18044,  1334,   119, 26876,  8125,  7174,  1110,  1632,
         1107,  1142,  1105,   146,   112,   182,  5733,  2903,  1451,  2523,
         1114,  1123,  1107,  1115,   146,  1169,  1243,  1139,  1493,  1113,
          119,  2119,  3869,   170,  1440,   119,   102,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,

#### **Define the Tokenization and Dataloaders Based on Hugging Face**

In [15]:
def tokenize_text (data):
    return tokenizer(
        data['text'],
        max_length = 256,
        truncation = True,
        padding = 'max_length'
    )

In [16]:
dataset = dataset.map(tokenize_text,batched=True)
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})

In [17]:
dataset = dataset.rename_column('label','labels')
dataset.set_format(type='torch', columns=['labels', 'input_ids', 'attention_mask'])

In [18]:
dataset.column_names

{'train': ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
 'test': ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
 'unsupervised': ['text',
  'labels',
  'input_ids',
  'token_type_ids',
  'attention_mask']}

##### **Define Dataloaders**

In [19]:
train_dataloader = DataLoader(dataset['train'], batch_size=4, shuffle=True)
test_dataloader = DataLoader(dataset['test'], batch_size=4, shuffle=False)

In [20]:
batch = next(iter(train_dataloader))
print(batch, batch['input_ids'].shape)

{'labels': tensor([0, 0, 0, 1]), 'input_ids': tensor([[ 101,  146,  112,  ...,    0,    0,    0],
        [ 101, 1188, 2523,  ..., 1231, 3155,  102],
        [ 101,  146, 1486,  ..., 1108, 1145,  102],
        [ 101,  138,  139,  ...,    0,    0,    0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]])} torch.Size([4, 256])


### **Custom Model Build**

##### **Text Embedding**

In [21]:
class TextEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, dropout ):
        super().__init__()
        self.d_model = d_model
        self.dropout = nn.Dropout(dropout)
        
        self.embedding = nn.Embedding(vocab_size,d_model)
        
    def forward(self, input_tokens):
        
        #input_tokens -> [batch_size, seq_len]
        embed_out = self.embedding(input_tokens) * math.sqrt(self.d_model)
        return self.dropout(embed_out)

#### **Positional Embeddings**

In [22]:
class PositionalEmbeddings (nn.Module):
    
    def __init__(self, max_seq_len, d_model, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        positions = torch.arange(0, max_seq_len).unsqueeze(1)
        
        pe =  torch.zeros(max_seq_len, d_model)
        
        div_term = torch.exp(
            torch.arange(0,d_model,2).float() * (-math.log(10000)/ d_model)
        )
        
        pe[:,0::2] = torch.sin(positions * div_term)
        pe[:,1::2] = torch.cos(positions * div_term)
        
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)
        
    def forward(self, word_embeddings):
        
        #word_embeddings -> [batch_size, seq_len, d_model]
        word_embedding_size = word_embeddings.size(1)
        
        pos_embeddings = word_embeddings + self.pe[:,0:word_embedding_size,:]
        return self.dropout(pos_embeddings)
        
        

#### **Custom Model Class**

In [23]:
class CustomClassificationModel (nn.Module):
    
    def __init__(self, vocab_size, d_model, 
                n_layers, n_heads, dropout, dim_feedforward, max_seq_len = 1000):
        super().__init__()
        self.d_model = d_model
        self.embeddings = TextEmbedding(vocab_size,d_model, dropout)
        self.pos_embeddings = PositionalEmbeddings(max_seq_len,d_model,dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(d_model,n_heads,dropout=dropout,batch_first=True,dim_feedforward=dim_feedforward)
        
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer=encoder_layer,
                                                        num_layers=n_layers)
        
        self.linear_layer = nn.Linear(in_features=d_model, out_features=2)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input_ids, attention_mask = None):
        
        embed_out = self.embeddings(input_ids) 
        pos_out = self.pos_embeddings(embed_out)
        
        src_key_padding_mask = (attention_mask == 0) if attention_mask is not None else None
        
        transformer_out = self.transformer_encoder(pos_out, src_key_padding_mask = src_key_padding_mask)
        transformer_out = transformer_out.mean(dim = 1)
        
        transformer_out = self.dropout(transformer_out)
        classify_out = self.linear_layer(transformer_out)
        
        return classify_out

#### **Model Instance Define**

In [24]:
vocab_size = tokenizer.vocab_size
n_heads = 4
d_model = 256
n_layers = 4
dropout = 0.1
dim_feedforward = 512

In [25]:
classify_model = CustomClassificationModel(vocab_size=vocab_size,d_model=d_model,n_layers=n_layers,
                                           n_heads=n_heads, dropout=dropout,
                                           dim_feedforward=dim_feedforward)

In [26]:
classify_model.to(device)

CustomClassificationModel(
  (embeddings): TextEmbedding(
    (dropout): Dropout(p=0.1, inplace=False)
    (embedding): Embedding(28996, 256)
  )
  (pos_embeddings): PositionalEmbeddings(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (linear_layer): Linear(in_features=256, out_

#### **Define Model Predict** 

In [27]:
def model_predict (model:CustomClassificationModel, input_sentence, tokenizer:AutoTokenizer):
    
    with torch.no_grad():
        
        tokens = tokenizer(input_sentence, return_tensors='pt', truncation=True).to(device)
        input_ids = tokens['input_ids']
        attention_mask = tokens['attention_mask']
        
        output = model(input_ids, attention_mask)
        predict = torch.argmax(output, dim=-1)
        
        predict_label = imdb_labels[predict.item()]
        
        return predict_label

In [28]:
sample = "I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967."
predict_label = model_predict(classify_model,sample, tokenizer)
predict_label

'negative review'

#### **Define Model Evaluation**

In [ ]:
def model_eval(model:CustomClassificationModel, dataloader:DataLoader):
    
    total_accuracy = []
    loss_epochs = []
    
    model.eval()
    
    with torch.no_grad():
        
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            label = batch['labels']
            input_ids = batch['input_ids']
            attention_mask = batch['attention_mask']
            
            output = model(input_ids, attention_mask)
            predict = torch.argmax()

In [42]:
for batch in test_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    label = batch['labels']
    input_ids = batch['input_ids']
    attention_mask = batch['attention_mask']
    print(label)

tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0], device='cuda:0')
tensor([0, 0

KeyboardInterrupt: 